# Colombian Goods Export News: Sentiment & Uncertainty

Study real Spanish-language news about Colombian **goods exports** to the world from March 2020 onward. The general monthly series comes first; destination countries are a secondary view. There is one subject: exports.

Measurement reference: [Mora-Quiñones, Orozco-Gallo & Mora-Pérez (2026), BE1340](https://doi.org/10.32468/be.1340). This project uses the complete Annex 3 dictionary and sentiment/uncertainty formulas on retrieved article text. It adapts the corpus and documents unspecified processing choices; it does not reproduce the Bank’s curated dataset or forecasting results. Automatic corpus/country proposals require human review.

## 1. Prepare Python
The next installation cell requires network access. All subsequent feature-analysis cells run on the committed snapshot without collecting news.

This is a research code sample, not a validated export forecast. Conservative scope screening supersedes the previous 696-article sample. See docs/RESEARCH_BRIEF.md for the research question, document-role audit and empirical validation design.


In [ ]:
%pip install -q pandas==2.3.3 plotly==6.9.0 streamlit==1.67.0 trafilatura==2.3.1 googlenewsdecoder==0.2.2


## 2. Load the private project
Download this repository using GitHub **Code → Download ZIP** while signed in, then upload the ZIP in Colab. No password or token belongs in the notebook. In local Jupyter, open the notebook from the repository root.


In [ ]:
from pathlib import Path
import io
import os
import sys
import tempfile
import zipfile

try:
    from google.colab import files
except ImportError:
    repo = Path.cwd()
else:
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one project ZIP")
    destination = Path(tempfile.mkdtemp(prefix="trade_news_"))
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        if sum(item.file_size for item in archive.infolist()) > 50_000_000:
            raise ValueError("Unexpectedly large project archive")
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Symbolic links are not accepted")
        archive.extractall(destination)
    roots = [path.parent for path in destination.rglob("README.md") if (path.parent / "trade_news").is_dir()]
    if len(roots) != 1:
        raise ValueError("Could not identify the project root")
    repo = roots[0]

assert (repo / "trade_news").is_dir(), "Run from the repository directory"
os.chdir(repo)
sys.path.insert(0, str(repo))
print("Project loaded:", repo.name)


## 3. Check real source lineage and reproduce feature aggregates
The frozen window covers March 2020–October 2026. A publisher-reported publication date determines the month. Missing bodies or dates never fall back to headlines. The repository shares source links, hashes and derived counts; publisher bodies stay in the ignored local archive. Therefore this offline cell reproduces decisions and arithmetic, not source-to-text extraction. The latter requires the archive or a new retrieval.

Historical discovery is incomplete. Analdex public archive posts supplement RSS discovery and dominate earlier periods. Source mix is not uniform; consult the monthly composition chart and snapshot report before interpreting trends.


In [ ]:
import json
import pandas as pd
import plotly.express as px
from trade_news.articles import read_jsonl
from trade_news.bank_method import aggregate, load_reference
from trade_news.corpus import apply_reviews, read_reviews, evaluate_sentiment

data = repo/"data"
raw = read_jsonl(data/"headlines_raw.jsonl")
manifest = read_jsonl(data/"article_manifest.jsonl")
log = json.loads((data/"collection_log.json").read_text("utf8"))
reviews = read_reviews(data/"annotations.csv")
rows = apply_reviews(manifest, reviews)
months = [request["month"] for request in log["requests"]]
indices = {"provisional": aggregate(rows, months), "human_reviewed": aggregate(rows, months, True)}
assert rows == read_jsonl(data/"headlines.jsonl")
assert indices == json.loads((data/"indices.json").read_text("utf8"))
lexicon, stopwords = load_reference()
print("Candidates:", len(raw), "| Retrieved article texts:", sum(r["fetch_status"] == "article_extracted" for r in rows),
      "| Eligible provisional articles:", sum(r["eligible_for_index"] for r in rows))
print("Published dictionary:", len(lexicon), "forms | Frozen stopwords:", len(stopwords))
print("Months with eligible text:", len({r["month"] for r in rows if r["eligible_for_index"]}), "/", len(months))
display(pd.DataFrame(rows)["fetch_status"].value_counts().rename("records").reset_index())


## 4. Inspect article evidence
The dictionary contains 188 positive and 376 negative forms. “Duplican” and “triplicaron” are absent; no new words are added to force a desired score. Inspect actual matched occurrences and article links. No polarity matches means an abstention, not neutral. The classifier’s scope and country proposals remain separate from human decisions.


In [ ]:
eligible = [r for r in rows if r["eligible_for_index"]]
display(pd.DataFrame(eligible)[["title", "publisher", "publication_date", "review_status", "article_url",
                                      "positive_count", "negative_count", "sentiment_score", "uncertainty_count"]].head(15))
if eligible:
    example = eligible[0]
    print(json.dumps({key: example.get(key) for key in ["article_url", "date_source", "text_sha256", "processed_tokens",
        "matched_terms", "scope_reason"]}, ensure_ascii=False, indent=2))


## 5. General monthly index
Pool word occurrences **before dividing**: `SENT=(P-N)/(P+N)`; `UNCERT=1000*U/W`. Uncertainty uses the six published forms of `incertidumbre` and `incierto`. Show article counts with the index. No articles or no polarity denominator means no estimate; gaps are not connected. News tone is not a measure of export performance.


In [ ]:
monthly = pd.DataFrame([r for r in indices["provisional"] if r["activity"] == "All exports"])
from trade_news.bank_method import trailing_quarterly_mean
display(monthly)
for column, title in [("sentiment", "Colombian goods export news: provisional sentiment"),
                       ("uncertainty_per_1000", "Uncertainty words per 1,000 processed words")]:
    fig = px.line(monthly, x="month", y=column, markers=True, title=title,
                  hover_data=["articles", "publishers", "positive_count", "negative_count", "status"])
    fig.update_traces(connectgaps=False, line_color="#abb2b8")
    fig.add_scatter(x=monthly["month"], y=trailing_quarterly_mean(monthly.to_dict("records"), column), mode="lines", name="Trailing 3-month mean", line={"color": "#142330"})
    if column == "sentiment":
        fig.update_yaxes(range=[-1.05, 1.05])
    fig.show()
px.bar(monthly, x="month", y="articles", title="Eligible article volume: coverage context").show()


## 6. Country evidence and actual review status
The country-only dashboard explains each destination link and offers an independent review form. The sparse activity view and regional/bloc controls have been retired. A country mentioned as a competitor is not automatically a buyer. Country review and sentiment review are separate; neither reference set is filled with model-generated labels.

The next cell reads actual saved assignments and current validation status. Metrics remain missing until genuine human reference judgments exist. Country metrics evaluate automatic predictions before manual corrections. See `docs/DESTINATIONS.md` and `docs/VALIDATION.md`.


In [ ]:
from trade_news.country_review import assignments, country_coverage, read_country_reviews, evaluate_countries
country_features = read_jsonl(data/"destinations.jsonl")
country_reviews = read_country_reviews(data, rows)
country_rows = assignments(rows, country_features, country_reviews)
display(pd.DataFrame(country_coverage(country_rows)))
print(json.dumps(evaluate_countries(rows, country_features, country_reviews), indent=2))
evaluation = evaluate_sentiment(rows, reviews)
print(json.dumps(evaluation, ensure_ascii=False, indent=2))
reviewed_series = pd.DataFrame([r for r in indices["human_reviewed"] if r["activity"] == "All exports"])
display(reviewed_series)


## 7. Optional new collection and article retrieval
Disabled by default. Public retrieval may fail or change; it is not guaranteed to recreate the frozen corpus. Use a new directory to preserve existing data and review work. Raw publisher HTML/text stays local and is ignored by Git. Do not bypass access restrictions. This step is separate from the offline feature analysis above.


In [ ]:
RUN_NEW_COLLECTION = False
if RUN_NEW_COLLECTION:
    from trade_news.collect import collect
    from trade_news.articles import acquire
    from trade_news.corpus import build
    fresh = repo/"data_new"
    if fresh.exists():
        raise ValueError("Choose a new directory rather than overwrite an existing snapshot")
    collect("2020-03-01", "2026-11-01", fresh)
    acquire(fresh)
    new_rows, new_indices = build(fresh, verify_local_text=True)


## 8. Academic interpretation
The original study’s exact stopword file and tokenizer code were not supplied; this project freezes NLTK Spanish stopwords and documents a Unicode tokenizer. Human corpus review, extraction inspection and independent sentiment labels are pending. Ten publication months are not a reproduction of the paper’s econometric validation. No nowcasting improvement, causal finding or official Bank endorsement is claimed. See `docs/METHODOLOGY.md`, `docs/RESEARCH_REFERENCE.md` and `docs/THIRD_PARTY_NOTICES.md`.
